# MRI-scans voorbereiden en indelen op aanzicht

Dit notebook filtert afbeeldingen op afmetingen en gebruikt handmatig gelabelde scans om aanzichten te herkennen. Het traint aparte self-trainingmodellen voor `Training` en `Testing` en deelt de afbeeldingen daarna in op **axiaal**, **sagittaal** en **coronaal**.

**Invoer:** `./Training/`, `./Testing/` en `./labeled_scans.csv` (kolommen `image`, `choice`).

**Uitvoer:** voorspellingen in CSV en kopieën van de afbeeldingen onder `./Sorted/Training/` en `./Sorted/Testing/`. De oorspronkelijke bestanden blijven ongewijzigd.

**Belangrijk:** het model voorspelt MRI-aanzichten, geen tumortype. De Testing-afbeeldingen worden als ongelabelde data gebruikt door een apart self-trainingmodel. Train een later tumormodel alleen op `Training` en bewaar de tumorlabels van `Testing` voor de uiteindelijke test.

## Installatie

Installeert de Python-pakketten die nodig zijn voor het inlezen, bewerken, trainen en beoordelen van het model.

In [1]:
# Stap 0: Installatie
%pip install torch
%pip install torch torchvision
%pip install numpy matplotlib scikit-learn
%pip install torch torchvision pandas pillow scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\achah\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\achah\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\achah\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\achah\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Imports

Laadt de gebruikte bibliotheken.

In [ ]:
# Stap 1: Imports
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from pathlib import Path
from torchvision import transforms
from torchvision.datasets import ImageFolder as ImageFolder
from torch.utils.data import Dataset, DataLoader
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.ensemble import RandomForestClassifier

## Afbeeldingen selecteren

`GefilterdeTumorDataset` doorzoekt de directe submappen van `Training` of `Testing`. Alleen PNG/JPG-afbeeldingen waarvan beide afmetingen minstens 256 pixels zijn, worden meegenomen. De bestaande submapnaam is hier **niet** het aanzichtlabel.

In [3]:
# Stap 3: Afbeeldingen selecteren
class GefilterdeTumorDataset(Dataset):
    def __init__(self, root, min=0, transform=None): 
        # Haal de submappen op als klassen
        self.classes = [
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        ]

        self.images = []
        self.transform = transform

        for index, group in enumerate(self.classes):
            group_map = os.path.join(root, group)

            for filename in os.listdir(group_map):
                if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    pad = os.path.join(group_map, filename)

                    # Controleer de afmetingen
                    with Image.open(pad) as img:
                        width, height = img.size

                    if width >= min and height >= min:
                        self.images.append((pad, index))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        pad, label = self.images[idx]
        img = Image.open(pad).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label


## Transformaties

Afbeeldingen worden voor de PyTorch-dataloader naar 512 × 512 pixels geschaald en omgezet naar tensors. Het classificatiemodel gebruikt later een aparte 64 × 64-grijswaardenrepresentatie.

In [4]:
# Stap 4: Transformaties
transform = transforms.Compose([
    transforms.ToTensor()
])

## Trainingsafbeeldingen inladen

Laadt de geselecteerde afbeeldingen uit `./Training`.

In [5]:
# Stap 5: Trainingsafbeeldingen inladen
dataset_train = GefilterdeTumorDataset(root='./Training', min=0,transform=transform)
dataloader_train = DataLoader(dataset_train, batch_size=32, shuffle=True)

print(f"Totaal aantal gefilterde foto's ingeladen: {len(dataset_train)}")

Totaal aantal gefilterde foto's ingeladen: 2870


## Testafbeeldingen inladen

Laadt ook `./Testing` in. Deze map wordt later volledig voorspeld en afzonderlijk ingedeeld.

In [6]:
# Stap 6: Testafbeeldingen inladen
dataset_test = GefilterdeTumorDataset(root='./Testing', min=0,transform=transform)
dataloader_test = DataLoader(dataset_test, batch_size=32, shuffle=True)

print(f"Totaal aantal gefilterde foto's ingeladen: {len(dataset_test)}")

Totaal aantal gefilterde foto's ingeladen: 394


## Import van handmatige labels uit CSV 

`labeled_scans.csv` bevat de kolommen `image` en `choice`. Alleen `Axiaal`, `Sagittaal` en `Coronaal` gelden als handmatige labels; andere afbeeldingen krijgen tijdelijk label `-1`.

In [7]:
# Stap 7: Handmatige labels uit CSV lezen
csv_path = "./labeled_scans.csv"

df = pd.read_csv(csv_path)

# Bestandsnamen uit de image-kolom halen
df["filename"] = df["image"]
df["label"] = df["choice"]

label_mapping = {
    "Axiaal": 0,
    "Sagittaal": 1,
    "Coronaal": 2
}

# Alleen daadwerkelijk gelabelde afbeeldingen
df_labeled = df[df["label"].isin(label_mapping)].copy()

# Numerieke labels toevoegen
df_labeled["label_id"] = df_labeled["label"].map(label_mapping)

print("Aantal CSV-rijen:", len(df))
print("Aantal gelabelde CSV-rijen:", len(df_labeled))

print("\nVerdeling van de labels:")
print(df_labeled["label"].value_counts())

Aantal CSV-rijen: 159
Aantal gelabelde CSV-rijen: 159

Verdeling van de labels:
label
Coronaal     54
Sagittaal    53
Axiaal       52
Name: count, dtype: int64


## Trainingsfeatures maken (prep)

Elke geselecteerde trainingsafbeelding wordt omgezet naar een genormaliseerde, afgevlakte 64 × 64-grijswaardenvector (4096 getallen). Labels worden op **bestandsnaam** gekoppeld.

In [ ]:


# Handmatig gelabelde afbeeldingen koppelen aan labels
label_lookup = dict(
    zip(df_labeled["filename"], df_labeled["label_id"])
)

# --- 1. De generieke functie ---
def get_all_images(image_paths, label_dict):
    """
    Haalt features (X), labels (y) en bestandsnamen op uit afbeeldingspaden.
    """
    X_temp = [] 
    y_temp = [] 
    filenames_temp = []

    # Loop door ALLE afbeeldingen in de meegegeven paden
    for path, _ in image_paths:
        filename = os.path.basename(path)
        
        # Bestaand label gebruiken, anders -1
        label = label_dict.get(filename, -1)
        
        with Image.open(path) as img:
            img = img.convert("L").resize((64, 64))
            img_array = np.asarray(img, dtype=np.float32).flatten() / 255.0
            
        X_temp.append(img_array)
        y_temp.append(label)
        filenames_temp.append(filename)
        
    # Converteer naar NumPy arrays en stuur ze als output terug
    X_out = np.array(X_temp, dtype=np.float32)
    y_out = np.array(y_temp, dtype=int)
    filenames_out = np.array(filenames_temp)
    
    return X_out, y_out, filenames_out

# --- 2. Toepassen op je TRAIN data ---
X_train, y_train, filenames_train = get_all_images(dataset_train.images, label_lookup)

print("--- Train Dataset Controle ---")
print("Totaal aantal afbeeldingen:", len(X_train))
print("Aantal handmatig gelabeld:", np.sum(y_train != -1))
print("Aantal ongelabeld:", np.sum(y_train == -1))
print("Verdeling labels:", np.unique(y_train, return_counts=True))

# Testing blijft ongelabeld; de labels uit de CSV worden alleen voor Training gebruikt.
X_test, y_test, filenames_test = get_all_images(dataset_test.images, {})

print("\n--- Test Dataset Controle ---")
print("Totaal aantal afbeeldingen:", len(X_test))
print("Aantal handmatig gelabeld:", np.sum(y_test != -1))
print("Aantal ongelabeld:", np.sum(y_test == -1))
print("Verdeling labels:", np.unique(y_test, return_counts=True))

## Gelabelde en ongelabelde data klaarzetten

Alle handmatig gelabelde voorbeelden uit `Training` blijven beschikbaar als startlabels. Het Training-model gebruikt daarnaast alleen ongelabelde afbeeldingen uit `Training`; het aparte Testing-model gebruikt dezelfde startlabels plus alleen ongelabelde afbeeldingen uit `Testing`.

In [ ]:
mask_labeled_training = y_train != -1
X_labeled_training = X_train[mask_labeled_training]
y_labeled_training = y_train[mask_labeled_training]
X_unlabeled_training = X_train[~mask_labeled_training]

# y_test is bewust volledig -1: Testing krijgt geen handmatige labels uit de CSV.
X_unlabeled_testing = X_test[y_test == -1]

if len(X_labeled_training) == 0:
    raise ValueError("Geen handmatig gelabelde Training-afbeeldingen gevonden.")

print("Handmatig gelabelde Training-afbeeldingen:", len(X_labeled_training))
print("Ongelabelde Training-afbeeldingen:", len(X_unlabeled_training))
print("Ongelabelde Testing-afbeeldingen:", len(X_unlabeled_testing))

## Semi-supervised Random Forest

Er worden twee aparte self-trainingmodellen gemaakt. Het model voor `Testing` start met de handmatig gelabelde voorbeelden uit `Training` en gebruikt uitsluitend de ongelabelde afbeeldingen uit `Testing` als extra data.

Dit model voorspelt het **aanzicht** (axiaal, sagittaal of coronaal), niet het tumortype. De 64 × 64-pixels leveren 4096 kenmerken per afbeelding op; dat zijn geen 4096 afbeeldingen.

In [ ]:
# Bomen mogen diep groeien om de kleine set handmatige labels goed te leren.
# Dit garandeert niet dat voorspellingen of pseudo-labels allemaal juist zijn.
def make_view_model(random_state):
    return SelfTrainingClassifier(
        estimator=RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=1,
            max_depth=None,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        ),
        threshold=0.90,
        max_iter=20,
    )

Model trainen - Training

In [ ]:
X_training_pool = np.concatenate([X_labeled_training, X_unlabeled_training])
y_training_pool = np.concatenate([
    y_labeled_training,
    np.full(len(X_unlabeled_training), -1, dtype=int),
])

training_view_model = make_view_model(random_state=42)
training_view_model.fit(X_training_pool, y_training_pool)

training_pseudo_labels = np.sum(
    (y_training_pool == -1) & (training_view_model.transduction_ != -1)
)
print("Training-model:", len(X_training_pool), "afbeeldingen")
print("Training-pseudo-labels toegevoegd:", training_pseudo_labels)
print("Iteraties:", training_view_model.n_iter_)
print("Stopreden:", training_view_model.termination_condition_)

In [ ]:
# Elke Training-afbeelding staat precies één keer in deze trainingspool.
if len(X_training_pool) != len(X_train) or len(y_training_pool) != len(X_training_pool):
    raise ValueError("De Training-pool bevat dubbele of ontbrekende afbeeldingen/labels.")
print("Controle Training-pool geslaagd:", len(X_training_pool), "afbeeldingen")

Model trainen - Testing apart, met Training-labels als start

In [ ]:
# Dit is een tweede model: Training-labels zijn de start, alleen Testing-beelden
# worden in deze fit als ongelabelde pool toegevoegd.
X_testing_pool = np.concatenate([X_labeled_training, X_unlabeled_testing])
y_testing_pool = np.concatenate([
    y_labeled_training,
    np.full(len(X_unlabeled_testing), -1, dtype=int),
])

testing_view_model = make_view_model(random_state=43)
testing_view_model.fit(X_testing_pool, y_testing_pool)

testing_pseudo_labels = np.sum(
    (y_testing_pool == -1) & (testing_view_model.transduction_ != -1)
)
print("Testing-model: Training-startlabels +", len(X_unlabeled_testing), "Testing-afbeeldingen")
print("Testing-pseudo-labels toegevoegd:", testing_pseudo_labels)
print("Iteraties:", testing_view_model.n_iter_)
print("Stopreden:", testing_view_model.termination_condition_)

if len(X_testing_pool) != len(X_labeled_training) + len(X_unlabeled_testing):
    raise ValueError("De Testing-pool bevat dubbele of ontbrekende afbeeldingen.")

## Geen modelscore berekenen

Deze notebook is nu uitsluitend bedoeld om aanzichtlabels te maken en de afbeeldingen visueel te controleren. Er wordt hier geen accuracy of testscore berekend. De `Testing`-afbeeldingen doen wel mee als ongelabelde data in hun eigen self-trainingmodel.

In [ ]:
# Stap 14: Model evalueren
print("Training-labels worden gebruikt voor de Training- en Testing-modellen.")
print("Handmatige controle gebeurt in Sorted/Training en Sorted/Testing.")

## Voorspellingen voor Training en Testing opslaan
 
Elk bestand wordt voorspeld met het eigen model: `training_view_model` voor `Training` en `testing_view_model` voor `Testing`. Per bronmap wordt een CSV opgeslagen met het relatieve bronpad, zodat gelijke bestandsnamen in verschillende submappen herkenbaar blijven.

De sorteeractie leest deze CSV-bestanden en **kopieert** de originele afbeeldingen naar aparte mappen voor `axiaal`, `sagittaal` en `coronaal`. Ook handmatig gelabelde trainingsafbeeldingen krijgen hier een modelvoorspelling, zodat de werkwijze voor beide datasets gelijk is. De originele bestanden worden niet verplaatst of verwijderd.

**Let op:** alleen afbeeldingen die aan de eerdere minimale afmeting voldoen, worden meegenomen. Self-training voegt alleen labels toe boven de confidence-drempel; `.predict()` geeft wel voor iedere afbeelding een aanzicht dat je handmatig kunt controleren. Een overfit model garandeert geen juiste voorspellingen.

In [ ]:
# Zelfde preprocessing als tijdens de training van het classificatiemodel.

label_names = {
    0: "Axiaal",
    1: "Sagittaal",
    2: "Coronaal"
}
# Gebruik de features van de twee aparte datasets en het bijbehorende model.
predict_csvs = {}
prediction_sources = (
    ("Training", dataset_train, X_train, training_view_model),
    ("Testing", dataset_test, X_test, testing_view_model),
)
for datasetname, dataset, features, view_model in prediction_sources:
    rootdirectory = Path(datasetname)
    paths = [Path(path) for path, _ in dataset.images]

    if not paths:
        print(f"{datasetname}: geen geschikte afbeeldingen gevonden; overgeslagen.")
        continue

    predicted_ids = view_model.predict(features)
    table = pd.DataFrame({
        "image": [str(path.relative_to(rootdirectory)) for path in paths],
        "predicted_view": [label_names[int(label)] for label in predicted_ids],
    })

    csv_path = Path(f"predicted_view{datasetname.lower()}.csv")
    table.to_csv(csv_path, index=False)
    predict_csvs[datasetname] = csv_path

    # check prints
    print(f"{datasetname}: {len(table)} voorspellingen opgeslagen in {csv_path}")
    print(table["predicted_view"].value_counts().to_string())

### Afbeeldingen naar mappen per aanzicht kopiëren

De CSV is leidend voor de indeling. De uitvoerstructuur wordt bijvoorbeeld `Sorted/Testing/axiaal/...`. De oorspronkelijke submapstructuur blijft **binnen** de aanzichtmap behouden om overschrijven bij gelijke bestandsnamen te vermijden. Opnieuw uitvoeren werkt de kopieën bij, maar verwijdert geen oude uitvoerbestanden.

In [ ]:
outputmap = Path("Sorted")
valid_views = {"axiaal", "sagittaal", "coronaal"}

for datasetname, csv_path in predict_csvs.items():

    rootdirectory = Path(datasetname).resolve()
    table = pd.read_csv(csv_path)

    # Maak alle drie de aanzichtmappen, ook als een klasse geen voorspellingen heeft.
    for view in valid_views:
        (outputmap / datasetname / view).mkdir(parents=True, exist_ok=True)
 
    count = 0
    for row in table.itertuples(index=False):
        view = str(row.predicted_view).strip().lower()

        if view not in valid_views:
            raise ValueError(f"Onbekend aanzicht in {csv_path}: {view}")

        relative_path = Path(str(row.image))
        rootfile = (rootdirectory / relative_path).resolve()

        # Bescherm tegen onbedoelde verwijzingen buiten de bronmap.
        if not rootfile.is_relative_to(rootdirectory):
            raise ValueError(f"Ongeldig pad in CSV: {relative_path}")

        if not rootfile.is_file():
            raise FileNotFoundError(f"Bronafbeelding ontbreekt: {rootfile}")

        # Bepaal het doelpad op basis van het voorspelde aanzicht.
        targetfile = outputmap / datasetname / view / relative_path

        targetfile.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(rootfile, targetfile)  # kopiëren, niet verplaatsen

        count += 1

    # Check statement
    print(f"{datasetname}: {count} afbeeldingen gekopieerd naar {outputmap / datasetname}")